# 📘 Notebook 5 — Supervised Fine-Tuning & LoRA

**MiniMind Step-by-Step Series** | Milestone 5 of 6

In Notebook 4, we pretrained the model on raw text. Now we teach it to follow instructions through **Supervised Fine-Tuning (SFT)** and implement **LoRA** (Low-Rank Adaptation) for parameter-efficient fine-tuning.

**Learning Objectives:**
- Run full SFT on instruction data with label masking
- Understand why SFT trains only on assistant responses
- Implement LoRA from scratch (low-rank decomposition A·B)
- Apply LoRA to freeze base weights and train only adapters
- Compare full SFT vs LoRA: parameter count, memory, speed
- Save, load, and merge LoRA weights

In [ ]:
# === Recap: Complete code from Notebooks 1-4 (Config + Model + Tokenizer + Data + Pretrain) ===!pip install tokenizers torch transformers --quietimport torchimport torch.nn as nnimport torch.nn.functional as Fimport math, json, osfrom dataclasses import dataclassfrom torch.utils.data import Dataset, DataLoaderimport matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as pltdevice = 'cuda' if torch.cuda.is_available() else 'cpu'# --- Config ---@dataclassclass MiniMindConfig:    hidden_size: int = 768; num_hidden_layers: int = 8; num_attention_heads: int = 8    num_key_value_heads: int = 4; head_dim: int = 96; vocab_size: int = 6400    intermediate_size: int = 2048; max_position_embeddings: int = 32768    rms_norm_eps: float = 1e-6; rope_theta: float = 1e6; dropout: float = 0.0    hidden_act: str = 'silu'; flash_attn: bool = True; bos_token_id: int = 1; eos_token_id: int = 2# --- RMSNorm ---class RMSNorm(nn.Module):    def __init__(self, dim, eps=1e-5):        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))    def norm(self, x): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)    def forward(self, x): return (self.weight * self.norm(x.float())).type_as(x)# --- RoPE ---def precompute_freqs_cis(dim, end=32768, rope_base=1e6):    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))    t = torch.arange(end); freqs = torch.outer(t, freqs).float()    return torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1), torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):    def rotate_half(x): return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)    cos, sin = cos.unsqueeze(unsqueeze_dim), sin.unsqueeze(unsqueeze_dim)    return ((q * cos) + (rotate_half(q) * sin)).to(q.dtype), ((k * cos) + (rotate_half(k) * sin)).to(k.dtype)# --- Attention ---def repeat_kv(x, n_rep):    bs, slen, n_kv_heads, head_dim = x.shape    if n_rep == 1: return x    return x[:, :, :, None, :].expand(bs, slen, n_kv_heads, n_rep, head_dim).reshape(bs, slen, n_kv_heads * n_rep, head_dim)class Attention(nn.Module):    def __init__(self, config):        super().__init__()        self.n_local_heads = config.num_attention_heads        self.n_local_kv_heads = config.num_key_value_heads        self.n_rep = self.n_local_heads // self.n_local_kv_heads        self.head_dim = config.head_dim        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)        self.attn_dropout = nn.Dropout(config.dropout)        self.resid_dropout = nn.Dropout(config.dropout)        self.dropout = config.dropout        self.flash = hasattr(F, 'scaled_dot_product_attention') and config.flash_attn    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):        bsz, seq_len, _ = x.shape        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)        xq, xk = self.q_norm(xq), self.k_norm(xk)        cos, sin = position_embeddings        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)        if past_key_value is not None:            xk = torch.cat([past_key_value[0], xk], dim=1)            xv = torch.cat([past_key_value[1], xv], dim=1)        past_kv = (xk, xv) if use_cache else None        xq, xk, xv = xq.transpose(1, 2), repeat_kv(xk, self.n_rep).transpose(1, 2), repeat_kv(xv, self.n_rep).transpose(1, 2)        if self.flash and seq_len > 1 and past_key_value is None:            output = F.scaled_dot_product_attention(xq, xk, xv, dropout_p=self.dropout if self.training else 0.0, is_causal=True)        else:            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)            scores[:, :, :, -seq_len:] += torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)            if attention_mask is not None:                scores += (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9            output = self.attn_dropout(F.softmax(scores.float(), dim=-1).type_as(xq)) @ xv        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)        return self.resid_dropout(self.o_proj(output)), past_kv# --- FeedForward (SwiGLU) ---class FeedForward(nn.Module):    def __init__(self, config):        super().__init__()        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)    def forward(self, x): return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))# --- Transformer Block ---class MiniMindBlock(nn.Module):    def __init__(self, layer_id, config):        super().__init__()        self.self_attn = Attention(config)        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        self.mlp = FeedForward(config)    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):        residual = hidden_states        hidden_states, present_kv = self.self_attn(self.input_layernorm(hidden_states), position_embeddings, past_key_value, use_cache, attention_mask)        hidden_states = hidden_states + residual        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))        return hidden_states, present_kv# --- Full Model ---class MiniMindModel(nn.Module):    def __init__(self, config):        super().__init__()        self.config = config        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)        self.dropout = nn.Dropout(config.dropout)        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(config.num_hidden_layers)])        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=config.max_position_embeddings, rope_base=config.rope_theta)        self.register_buffer("freqs_cos", freqs_cos, persistent=False)        self.register_buffer("freqs_sin", freqs_sin, persistent=False)    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False):        batch_size, seq_length = input_ids.shape        past_key_values = past_key_values or [None] * len(self.layers)        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0        hidden_states = self.dropout(self.embed_tokens(input_ids))        position_embeddings = (self.freqs_cos[start_pos:start_pos + seq_length], self.freqs_sin[start_pos:start_pos + seq_length])        presents = []        for layer, past_kv in zip(self.layers, past_key_values):            hidden_states, present = layer(hidden_states, position_embeddings, past_key_value=past_kv, use_cache=use_cache, attention_mask=attention_mask)            presents.append(present)        return self.norm(hidden_states), presentsclass MiniMindForCausalLM(nn.Module):    def __init__(self, config):        super().__init__()        self.config = config        self.model = MiniMindModel(config)        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)        self.model.embed_tokens.weight = self.lm_head.weight    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, use_cache=False):        hidden_states, past_key_values = self.model(input_ids, attention_mask, past_key_values, use_cache)        logits = self.lm_head(hidden_states)        loss = None        if labels is not None:            shift_logits = logits[..., :-1, :].contiguous()            shift_labels = labels[..., 1:].contiguous()            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)        return {'loss': loss, 'logits': logits, 'past_key_values': past_key_values}# --- Synthetic Data ---pretrain_texts = [    "The transformer architecture was introduced in 2017.",    "Language models learn to predict the next token in a sequence.",    "Attention mechanisms allow models to focus on relevant parts of the input.",    "Neural networks consist of layers of interconnected nodes.",    "Gradient descent is the primary optimization algorithm for training neural networks.",    "Backpropagation computes gradients by applying the chain rule.",    "Tokenization converts raw text into numerical representations.",    "The softmax function converts logits into probability distributions.",    "Residual connections help train very deep networks.",    "Layer normalization stabilizes training by normalizing activations.",    "Dropout randomly zeroes activations to prevent overfitting.",    "The learning rate controls the step size during optimization.",    "Batch size determines how many samples are processed together.",    "Cross-entropy loss measures the difference between predicted and true distributions.",    "Weight decay adds L2 regularization to prevent overfitting.",    "Mixed precision training uses float16 to speed up computation.",    "The Adam optimizer combines momentum and adaptive learning rates.",    "Cosine annealing gradually reduces the learning rate during training.",    "Embedding layers map discrete tokens to continuous vectors.",    "Position embeddings encode the order of tokens in a sequence.",]os.makedirs('synthetic_data', exist_ok=True)pretrain_path = 'synthetic_data/pretrain.jsonl'with open(pretrain_path, 'w') as f:    for text in pretrain_texts:        f.write(json.dumps({'text': text}) + '\n')sft_conversations = [    [{"role": "user", "content": "What is a transformer?"}, {"role": "assistant", "content": "A transformer is a neural network architecture that uses self-attention mechanisms to process sequential data in parallel, rather than sequentially like RNNs."}],    [{"role": "user", "content": "Explain attention in neural networks."}, {"role": "assistant", "content": "Attention allows a model to dynamically focus on different parts of the input when producing each output element. It computes weighted sums of values based on query-key similarity scores."}],    [{"role": "user", "content": "What is tokenization?"}, {"role": "assistant", "content": "Tokenization is the process of breaking text into smaller units called tokens. Common methods include BPE (Byte-Pair Encoding), which iteratively merges frequent character pairs."}],    [{"role": "user", "content": "How does gradient descent work?"}, {"role": "assistant", "content": "Gradient descent updates model parameters by moving them in the direction that reduces the loss function. The step size is controlled by the learning rate."}],    [{"role": "user", "content": "What is overfitting?"}, {"role": "assistant", "content": "Overfitting occurs when a model learns the training data too well, including noise and outliers, and fails to generalize to new unseen data. Techniques like dropout and weight decay help prevent it."}],    [{"role": "user", "content": "Explain the softmax function."}, {"role": "assistant", "content": "Softmax converts a vector of real numbers into a probability distribution. Each output value is between 0 and 1, and all outputs sum to 1. It is commonly used in the final layer of classification models."}],    [{"role": "user", "content": "What is transfer learning?"}, {"role": "assistant", "content": "Transfer learning involves taking a model trained on one task and adapting it to a different but related task. This saves computation and often improves performance, especially with limited data."}],    [{"role": "user", "content": "What is a loss function?"}, {"role": "assistant", "content": "A loss function measures how far the model's predictions are from the true values. Common examples include cross-entropy for classification and mean squared error for regression."}],]sft_path = 'synthetic_data/sft.jsonl'with open(sft_path, 'w') as f:    for conv in sft_conversations:        f.write(json.dumps({'conversations': conv}) + '\n')# --- Tokenizer (BPE) ---from tokenizers import decoders, models, pre_tokenizers, trainers, TokenizerVOCAB_SIZE = 6400all_special = [    '<|endoftext|>', '<|im_start|>', '<|im_end|>', '<|im_sep|>',    '<|endofprefix|>', '<|startoftext|>', '<|extra_0|>', '<|extra_1|>',    '<|extra_2|>', '<|extra_3|>', '<|extra_4|>', '<|extra_5|>',    '<|extra_6|>', '<|extra_7|>', '<|extra_8|>', '<|extra_9|>',    '<pad>', '<mask>', '<sep>', '<cls>',    '<unused_0>', '<unused_1>', '<unused_2>', '<unused_3>',    '<unused_4>', '<unused_5>', '<unused_6>', '<unused_7>',    '<think>', '</think>', '<tool_call>', '</tool_call>',    '<tool_response>', '</tool_response>', '<code>', '</code>',]with open(pretrain_path) as f:    corpus = [json.loads(line)['text'] for line in f] * 100raw_tokenizer = Tokenizer(models.BPE())raw_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)trainer_obj = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=all_special, show_progress=False, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())raw_tokenizer.decoder = decoders.ByteLevel()raw_tokenizer.train_from_iterator(corpus, trainer=trainer_obj)class SimpleTokenizer:    def __init__(self, tokenizer, special_tokens):        self.tokenizer = tokenizer        vocab = tokenizer.get_vocab()        self.bos_token_id = vocab.get('<|im_start|>', 1)        self.eos_token_id = vocab.get('<|im_end|>', 2)        self.pad_token_id = vocab.get('<pad>', 0)    def __call__(self, text, add_special_tokens=False, max_length=None, truncation=False):        ids = self.tokenizer.encode(text).ids        if max_length and truncation: ids = ids[:max_length]        return type('Encoding', (), {'input_ids': ids})()    def decode(self, ids): return self.tokenizer.decode(ids)tokenizer = SimpleTokenizer(raw_tokenizer, all_special)# --- Chat Template ---def apply_chat_template(messages):    text = ''    for msg in messages:        text += f'<|im_start|>{msg["role"]}\n{msg["content"]}<|im_end|>\n'    return text# --- Datasets ---class PretrainDataset(Dataset):    def __init__(self, data_path, tokenizer, max_length=128):        super().__init__(); self.tokenizer = tokenizer; self.max_length = max_length; self.samples = []        with open(data_path) as f:            for line in f: self.samples.append(json.loads(line))    def __len__(self): return len(self.samples)    def __getitem__(self, index):        text = str(self.samples[index]['text'])        tokens = self.tokenizer(text, add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))        input_ids = torch.tensor(input_ids[:self.max_length], dtype=torch.long)        labels = input_ids.clone(); labels[input_ids == self.tokenizer.pad_token_id] = -100        return input_ids, labelsclass SFTDataset(Dataset):    def __init__(self, jsonl_path, tokenizer, max_length=256):        super().__init__(); self.tokenizer = tokenizer; self.max_length = max_length; self.samples = []        with open(jsonl_path) as f:            for line in f: self.samples.append(json.loads(line))        self.bos_marker = tokenizer(f'<|im_start|>assistant\n', add_special_tokens=False).input_ids        self.eos_marker = tokenizer(f'<|im_end|>\n', add_special_tokens=False).input_ids    def __len__(self): return len(self.samples)    def generate_labels(self, input_ids):        labels = [-100] * len(input_ids)        i = 0        while i < len(input_ids):            if input_ids[i:i + len(self.bos_marker)] == self.bos_marker:                start = i + len(self.bos_marker); end = start                while end < len(input_ids):                    if input_ids[end:end + len(self.eos_marker)] == self.eos_marker: break                    end += 1                for j in range(start, min(end + len(self.eos_marker), len(input_ids))): labels[j] = input_ids[j]                i = end + len(self.eos_marker) if end < len(input_ids) else len(input_ids)            else: i += 1        return labels    def __getitem__(self, index):        conversations = self.samples[index]['conversations']        prompt = apply_chat_template(conversations)        input_ids = self.tokenizer(prompt).input_ids[:self.max_length]        input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))        labels = self.generate_labels(input_ids)        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)# --- LR Schedule & Generate ---def get_lr(step, total_steps, warmup_steps, max_lr, min_lr=1e-6):    if step < warmup_steps: return max_lr * step / warmup_steps    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))@torch.no_grad()def generate(model, tokenizer, prompt_ids, max_new_tokens=50, temperature=1.0, top_k=0):    model.eval()    input_ids = torch.tensor([prompt_ids], dtype=torch.long).to(device)    for _ in range(max_new_tokens):        outputs = model(input_ids)        logits = outputs['logits'][:, -1, :] / temperature        if top_k > 0:            topk_values, _ = torch.topk(logits, top_k)            logits[logits < topk_values[:, -1:]] = float('-inf')        probs = F.softmax(logits, dim=-1)        next_token = torch.multinomial(probs, num_samples=1)        input_ids = torch.cat([input_ids, next_token], dim=-1)        if next_token.item() == tokenizer.eos_token_id: break    return input_ids[0].tolist()# --- Quick Pretrain Run ---config = MiniMindConfig()pretrain_ds = PretrainDataset(pretrain_path, tokenizer, max_length=64)sft_ds = SFTDataset(sft_path, tokenizer, max_length=128)model = MiniMindForCausalLM(config).to(device)optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)train_loader = DataLoader(pretrain_ds, batch_size=4, shuffle=True)total_steps = 3 * len(train_loader)warmup_steps = int(total_steps * 0.1)CHECKPOINT_DIR = 'minimind_checkpoints'os.makedirs(CHECKPOINT_DIR, exist_ok=True)model.train()for epoch in range(3):    for step, (input_ids, labels) in enumerate(train_loader):        input_ids, labels = input_ids.to(device), labels.to(device)        lr = get_lr(epoch * len(train_loader) + step, total_steps, warmup_steps, 5e-4)        for pg in optimizer.param_groups: pg['lr'] = lr        loss = model(input_ids, labels=labels)['loss']        loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        optimizer.step(); optimizer.zero_grad()    print(f'  Pretrain epoch {epoch+1}/3 — loss: {loss.item():.4f}')pretrain_ckpt = os.path.join(CHECKPOINT_DIR, 'pretrain_final.pth')torch.save(model.state_dict(), pretrain_ckpt)del model, optimizertorch.cuda.empty_cache() if torch.cuda.is_available() else Noneprint(f'\n✅ All prior code loaded — config: {config.hidden_size}d, {config.num_hidden_layers}L, {config.vocab_size}V')print(f'   Tokenizer: BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}')print(f'   Pretrain: {len(pretrain_ds)} samples, SFT: {len(sft_ds)} samples')print(f'   Pretrain checkpoint: {pretrain_ckpt}')

## Part A — Full Supervised Fine-Tuning (SFT)

SFT teaches a pretrained model to follow instructions by training on (question, answer) pairs. The key insight: **we only compute loss on assistant response tokens** — the model learns to answer, not to memorize questions.

This is "full" SFT because we update ALL model parameters.

In [ ]:
# === Full SFT Training ===SFT_LR = 1e-4SFT_EPOCHS = 3# Start from pretrained weightssft_model = MiniMindForCausalLM(config).to(device)sft_model.load_state_dict(torch.load(pretrain_ckpt, map_location=device, weights_only=True))sft_optimizer = torch.optim.AdamW(sft_model.parameters(), lr=SFT_LR, weight_decay=0.01)sft_ds = SFTDataset(sft_path, tokenizer, max_length=128)sft_loader = DataLoader(sft_ds, batch_size=4, shuffle=True)total_sft_steps = SFT_EPOCHS * len(sft_loader)warmup_sft = int(total_sft_steps * 0.1)sft_loss_history = []print(f'=== Full SFT Training ===')print(f'Trainable params: {sum(p.numel() for p in sft_model.parameters() if p.requires_grad):,}')print(f'Epochs: {SFT_EPOCHS}, Steps/epoch: {len(sft_loader)}')sft_model.train()for epoch in range(SFT_EPOCHS):    epoch_loss = 0.0    for step, (input_ids, labels) in enumerate(sft_loader):        input_ids, labels = input_ids.to(device), labels.to(device)        # Update LR        global_step = epoch * len(sft_loader) + step        lr = get_lr(global_step, total_sft_steps, warmup_sft, SFT_LR)        for pg in sft_optimizer.param_groups:            pg['lr'] = lr        outputs = sft_model(input_ids, labels=labels)        loss = outputs['loss']        loss.backward()        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)        sft_optimizer.step()        sft_optimizer.zero_grad()        sft_loss_history.append(loss.item())        epoch_loss += loss.item()    avg = epoch_loss / len(sft_loader)    print(f'  Epoch {epoch+1}/{SFT_EPOCHS} — avg loss: {avg:.4f}')# Save SFT checkpointsft_ckpt = os.path.join(CHECKPOINT_DIR, 'full_sft.pth')torch.save(sft_model.state_dict(), sft_ckpt)print(f'\n✅ Full SFT complete — final loss: {sft_loss_history[-1]:.4f}')

In [ ]:
# === ✅ Milestone 5A: SFT Training ===initial_sft = sum(sft_loss_history[:3]) / 3final_sft = sum(sft_loss_history[-3:]) / 3assert final_sft < initial_sft, 'SFT loss should decrease!'print(f'✅ Milestone 5A passed — SFT loss: {initial_sft:.4f} → {final_sft:.4f}')# Generate a responsesft_model.eval()prompt = apply_chat_template([{"role": "user", "content": "What is a transformer?"}])prompt_ids = tokenizer(prompt).input_idsgen_ids = generate(sft_model, tokenizer, prompt_ids, max_new_tokens=40, temperature=0.8)print(f'Q: What is a transformer?')print(f'A: {raw_tokenizer.decode(gen_ids[len(prompt_ids):])}')

## Part B — LoRA: Low-Rank Adaptation

**Problem:** Full SFT updates all parameters — expensive and requires storing a full copy of the model for each task.

**Solution:** LoRA decomposes weight updates into low-rank matrices:
$$W' = W + \Delta W = W + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$, with rank $r \ll d$.

**Benefits:**
- Only ~2-5% of parameters are trainable
- Original weights stay frozen
- Multiple LoRA adapters can share the same base model
- Adapters can be merged back into the base model

**Initialization:** A is Gaussian N(0, 0.02), B is zero → ΔW starts at zero (no change).

In [ ]:
# === LoRA Module ===class LoRA(nn.Module):    """Low-Rank Adaptation module."""    def __init__(self, in_features, out_features, rank=16):        super().__init__()        self.rank = rank        self.A = nn.Linear(in_features, rank, bias=False)   # Down-projection        self.B = nn.Linear(rank, out_features, bias=False)   # Up-projection        # Init: A ~ N(0, 0.02), B = 0 → ΔW starts at zero        self.A.weight.data.normal_(mean=0.0, std=0.02)        self.B.weight.data.zero_()    def forward(self, x):        return self.B(self.A(x))def apply_lora(model, rank=16):    """Add LoRA adapters to all square Linear layers."""    for name, module in model.named_modules():        if isinstance(module, nn.Linear) and module.weight.shape[0] == module.weight.shape[1]:            lora = LoRA(module.weight.shape[0], module.weight.shape[1], rank=rank).to(next(model.parameters()).device)            setattr(module, 'lora', lora)            original_forward = module.forward            def forward_with_lora(x, layer1=original_forward, layer2=lora):                return layer1(x) + layer2(x)            module.forward = forward_with_lora    return modeldef save_lora(model, path):    """Save only LoRA parameters."""    state_dict = {}    for name, module in model.named_modules():        if hasattr(module, 'lora'):            lora_state = {f'{name}.lora.{k}': v.cpu() for k, v in module.lora.state_dict().items()}            state_dict.update(lora_state)    torch.save(state_dict, path)    return len(state_dict)def load_lora(model, path):    """Load LoRA parameters."""    state_dict = torch.load(path, map_location=next(model.parameters()).device, weights_only=True)    for name, module in model.named_modules():        if hasattr(module, 'lora'):            lora_state = {k.replace(f'{name}.lora.', ''): v for k, v in state_dict.items() if f'{name}.lora.' in k}            module.lora.load_state_dict(lora_state)def merge_lora(model, lora_path, save_path):    """Merge LoRA weights into base model and save."""    load_lora(model, lora_path)    state_dict = {k: v.cpu() for k, v in model.state_dict().items() if '.lora.' not in k}    for name, module in model.named_modules():        if isinstance(module, nn.Linear) and hasattr(module, 'lora'):            state_dict[f'{name}.weight'] = (                module.weight.data.clone().cpu() +                (module.lora.B.weight.data @ module.lora.A.weight.data).cpu()            )    torch.save(state_dict, save_path)print('LoRA module defined ✓')

In [ ]:
# === Apply LoRA ===lora_model = MiniMindForCausalLM(config).to(device)lora_model.load_state_dict(torch.load(pretrain_ckpt, map_location=device, weights_only=True))# Apply LoRA adaptersapply_lora(lora_model, rank=16)# Freeze base weights, only train LoRAfor name, param in lora_model.named_parameters():    if 'lora' not in name:        param.requires_grad = False# Count parameterstotal_params = sum(p.numel() for p in lora_model.parameters())trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)frozen_params = total_params - trainable_paramsprint(f'=== LoRA Parameter Analysis ===')print(f'Total parameters:     {total_params:>12,}')print(f'Trainable (LoRA):     {trainable_params:>12,}')print(f'Frozen (base):        {frozen_params:>12,}')print(f'Trainable ratio:      {trainable_params/total_params*100:>11.2f}%')print(f'\nFull SFT trains:      {frozen_params:>12,} params')print(f'LoRA trains:          {trainable_params:>12,} params')print(f'Reduction:            {frozen_params/trainable_params:>11.1f}x fewer')

In [ ]:
# === LoRA Fine-Tuning ===LORA_LR = 1e-4LORA_EPOCHS = 3lora_optimizer = torch.optim.AdamW(    [p for p in lora_model.parameters() if p.requires_grad],    lr=LORA_LR, weight_decay=0.01)lora_loader = DataLoader(sft_ds, batch_size=4, shuffle=True)total_lora_steps = LORA_EPOCHS * len(lora_loader)warmup_lora = int(total_lora_steps * 0.1)lora_loss_history = []print(f'=== LoRA SFT Training ===')print(f'Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)')lora_model.train()for epoch in range(LORA_EPOCHS):    epoch_loss = 0.0    for step, (input_ids, labels) in enumerate(lora_loader):        input_ids, labels = input_ids.to(device), labels.to(device)        global_step = epoch * len(lora_loader) + step        lr = get_lr(global_step, total_lora_steps, warmup_lora, LORA_LR)        for pg in lora_optimizer.param_groups:            pg['lr'] = lr        outputs = lora_model(input_ids, labels=labels)        loss = outputs['loss']        loss.backward()        torch.nn.utils.clip_grad_norm_(lora_model.parameters(), 1.0)        lora_optimizer.step()        lora_optimizer.zero_grad()        lora_loss_history.append(loss.item())        epoch_loss += loss.item()    avg = epoch_loss / len(lora_loader)    print(f'  Epoch {epoch+1}/{LORA_EPOCHS} — avg loss: {avg:.4f}')# Save LoRA weightslora_save_path = os.path.join(CHECKPOINT_DIR, 'lora_weights.pth')num_saved = save_lora(lora_model, lora_save_path)lora_file_size = os.path.getsize(lora_save_path) / 1024print(f'\n✅ LoRA training complete — final loss: {lora_loss_history[-1]:.4f}')print(f'   Saved {num_saved} LoRA tensors → {lora_save_path} ({lora_file_size:.1f} KB)')

In [ ]:
# === ✅ Milestone 5B: LoRA vs Full SFT Comparison ===initial_lora = sum(lora_loss_history[:3]) / 3final_lora = sum(lora_loss_history[-3:]) / 3assert final_lora < initial_lora, 'LoRA loss should decrease!'print(f'✅ Milestone 5B passed — LoRA fine-tuning works')print(f'\n=== Comparison: Full SFT vs LoRA ===')print(f'{"Metric":<25s} {"Full SFT":>12s} {"LoRA":>12s}')print(f'{"-"*50}')full_params = sum(p.numel() for p in sft_model.parameters())print(f'{"Trainable params":<25s} {full_params:>12,} {trainable_params:>12,}')print(f'{"% of total":<25s} {100:>11.1f}% {trainable_params/total_params*100:>11.1f}%')print(f'{"Initial loss":<25s} {initial_sft:>12.4f} {initial_lora:>12.4f}')print(f'{"Final loss":<25s} {final_sft:>12.4f} {final_lora:>12.4f}')# Checkpoint size comparisonfull_size = os.path.getsize(sft_ckpt) / 1024lora_size = os.path.getsize(lora_save_path) / 1024print(f'{"Checkpoint size":<25s} {full_size:>10.0f} KB {lora_size:>10.0f} KB')print(f'{"Size reduction":<25s} {"1x":>12s} {full_size/lora_size:>10.0f}x')

## Part C — Merge & Deploy LoRA

LoRA adapters can be merged back into the base model weights:
$$W_{\text{merged}} = W_{\text{base}} + B \times A$$

After merging, the model runs at the same speed as the original — no LoRA overhead during inference.

In [ ]:
# === Merge LoRA Weights ===merged_path = os.path.join(CHECKPOINT_DIR, 'merged_model.pth')# Create fresh model, apply LoRA, then mergemerge_model = MiniMindForCausalLM(config).to(device)merge_model.load_state_dict(torch.load(pretrain_ckpt, map_location=device, weights_only=True))apply_lora(merge_model, rank=16)merge_lora(merge_model, lora_save_path, merged_path)# Load merged model (no LoRA needed!)final_model = MiniMindForCausalLM(config).to(device)final_model.load_state_dict(torch.load(merged_path, map_location=device, weights_only=True))# === ✅ Milestone 5C: Merged Model Works ===final_model.eval()prompt = apply_chat_template([{"role": "user", "content": "What is attention?"}])prompt_ids = tokenizer(prompt).input_idsgen_ids = generate(final_model, tokenizer, prompt_ids, max_new_tokens=40, temperature=0.8)print(f'✅ Milestone 5C passed — LoRA merged successfully')print(f'   Merged checkpoint: {os.path.getsize(merged_path)/1024:.0f} KB')print(f'\nQ: What is attention?')print(f'A: {raw_tokenizer.decode(gen_ids[len(prompt_ids):])}')

## 🎯 Notebook 5 Summary

| Component | Key Concept |
|---|---|
| **Full SFT** | Update all parameters on instruction data |
| **Label Masking** | Only train on assistant response tokens |
| **LoRA Module** | Low-rank A·B decomposition of weight updates |
| **apply_lora** | Add LoRA adapters to square Linear layers |
| **save/load_lora** | Save only adapter weights (~2-5% of total) |
| **merge_lora** | W_merged = W_base + B×A — no inference overhead |

**Milestones completed:** 5A (SFT training), 5B (LoRA comparison), 5C (LoRA merge)

### Next Notebook
In Notebook 6, we explore **evaluation and inference** — different decoding strategies, temperature effects, perplexity measurement, and multi-turn conversation.